# Importando as bibliotecas

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import current_timestamp
from pyspark.sql import functions as F
from pyspark.sql import Window as W
from pyspark.sql.functions import regexp_replace, col
from pyspark.sql.types import DecimalType
from pyspark.sql.functions import col, count, when, isnan
from functools import reduce

# Criando o database silver

In [0]:
%sql
USE CATALOG projeto;
-- DROP DATABASE silver CASCADE; -- Executar se tiver uma silver já criada
CREATE DATABASE IF NOT EXISTS silver;

## Tabela: Atendentes

## Tabela: Canais

## Tabela: Chamados

## Tabela: Chamados_Hora

In [0]:
# Lendo a tabela chamados_hora da camada bronze e visualizando os primeiros registros
df_bz = spark.table("bronze.chamados_hora")
print(f"bronze.chamados_hora: {df_bz.count()} rows")
display(df_bz.limit(10))

In [0]:
# Remoção de duplicatas:
df_max = (
    df_bz.groupBy("ID_Chamado")
         .agg(F.max("ingestion_timestamp").alias("ingestion_timestamp"))
)
df_bz = df_bz.join(df_max, on=["ID_Chamado", "ingestion_timestamp"], how="inner")


# Renomeando as colunas para snake_case
df = (
    df_bz
    .withColumnRenamed("ID_Chamado", "id_chamado")
    .withColumnRenamed("ID_Cliente", "id_cliente")
    .withColumnRenamed("Hora_Abertura_Chamado", "hora_abertura_chamado_raw")
    .withColumnRenamed("Hora_Inicio_Atendimento", "hora_inicio_atendimento_raw")
    .withColumnRenamed("Hora_Finalizacao_Atendimento", "hora_finalizacao_atendimento_raw")
    # ingestion_timestamp já está em snake_case
)

# Criando uma função para tirar o " �s " e converter pra timestamp
def to_ts(col):
    return F.to_timestamp(F.regexp_replace(col, " �s ", " "), "dd/MM/yyyy HH:mm:ss")

df = (
    # Aplicando a função nas três colunas de tempo e dropando as raws
    df
    .withColumn("data_hora_abertura", to_ts(F.col("hora_abertura_chamado_raw")))
    .withColumn("data_hora_inicio_atendimento", to_ts(F.col("hora_inicio_atendimento_raw")))
    .withColumn("data_hora_finalizacao_atendimento", to_ts(F.col("hora_finalizacao_atendimento_raw")))
    .drop(
        "hora_abertura_chamado_raw",
        "hora_inicio_atendimento_raw",
        "hora_finalizacao_atendimento_raw"
    )

    # garantir tipos dos IDs em long
    .withColumn("id_chamado", F.col("id_chamado").cast("long"))
    .withColumn("id_cliente", F.col("id_cliente").cast("long"))

    # Criando duas métricas úteis de minutos
    .withColumn(
        "tempo_espera_atendimento_min",
        F.round((F.col("data_hora_inicio_atendimento").cast("long") - F.col("data_hora_abertura").cast("long")) / 60.0, 2)
    )

    .withColumn(
        "tempo_atendimento_min",
        F.round(
            (F.col("data_hora_finalizacao_atendimento").cast("long") - F.col("data_hora_inicio_atendimento").cast("long")) / 60.0, 2)
    )
)

# Ordenando as colunas
df = df.select(
    "id_chamado",
    "id_cliente",
    "data_hora_abertura",
    "data_hora_inicio_atendimento",
    "data_hora_finalizacao_atendimento",
    "tempo_espera_atendimento_min",
    "tempo_atendimento_min",
    "ingestion_timestamp"
)

display(df.limit(10))


In [0]:
# Salvando no silver.dim_chamado_hora (formato delta por padrão)
df.write.mode("overwrite").saveAsTable("silver.dim_chamado_hora")
print(f"silver.dim_chamado_hora: {df.count()} rows")

## Tabela: Clientes

In [0]:
df_clientes_bronze = spark.table("bronze.clientes")
print(f"{df_clientes_bronze.count()} rows")
display(df_clientes_bronze.limit(10))

In [0]:

def analyze_missing_data(df_origem):
    # 1. Definir explicitamente as colunas alvo
    colunas_alvo = ["id_cliente", "nome", "email", "regiao", "idade"]
    
    print(f"Analisando apenas as colunas: {colunas_alvo}")
    
    df_check = df_origem.select(*colunas_alvo)
    total_rows = df_check.count()

    # 2. Preparar condições (seguro contra erro de tipo)
    expressoes_agg = []
    condicoes_filtro = []
    
    for c_name, c_type in df_check.dtypes:
        if c_type in ('double', 'float'):
            condicao = col(c_name).isNull() | isnan(c_name)
        else:
            condicao = col(c_name).isNull()
            
        expressoes_agg.append(count(when(condicao, c_name)).alias(c_name))
        condicoes_filtro.append(condicao)
    
    # 3. Exibir Resumo (Contagem e Porcentagem)
    resultado_agg = df_check.select(expressoes_agg).collect()[0].asDict()
    
    dados_resumo = []
    for coluna, qtd_missing in resultado_agg.items():
        dados_resumo.append({
            "Coluna": coluna,
            "Faltantes": qtd_missing,
            "%": round((qtd_missing / total_rows) * 100, 2)
        })
        
    print("--- Resumo de Faltantes ---")
    display(spark.createDataFrame(dados_resumo).orderBy(col("Faltantes").desc()))
    
    # 4. Mostrar linhas que possuem QUALQUER um dos campos acima nulo
    if condicoes_filtro:
        condicao_final = reduce(lambda x, y: x | y, condicoes_filtro)
        linhas_com_problema = df_check.filter(condicao_final)
        
        if linhas_com_problema.count() > 0:
            print("--- Amostra de linhas com dados faltando ---")
            display(linhas_com_problema.limit(10))
        else:
            print("Sucesso! Nenhuma linha com dados faltantes nessas colunas.")

df_clientes = spark.table("bronze.clientes")
analyze_missing_data(df_clientes)

In [0]:
#df_clientes = spark.table("bronze.clientes")

# 1. Tratamento de Região (Remover acentos e deixar maiúsculo)
acentos     = "áéíóúàèìòùâêîôûãõçÁÉÍÓÚÀÈÌÒÙÂÊÎÔÛÃÕÇ"
sem_acentos = "aeiouaeiouaeiouaocAEIOUAEIOUAEIOUAOC"

df_clientes = df_clientes.withColumn(
    "regiao_clean", 
    F.upper(F.translate(F.col("regiao"), acentos, sem_acentos))
)

# 2. Validação de E-mail (Regex simples) e Tratamento de Nomes
email_pattern = r"^[\w\.\-\+]+@[\w\.-]+\.\w+$"


df_clientes = (
    df_clientes
    .withColumn("id_cliente", F.col("id_cliente").cast("long"))
    .withColumn("idade", F.col("idade").cast("integer"))
    
    # Filtra ou marca e-mails inválidos (aqui optamos por manter apenas válidos na Silver, 
    # ou você pode criar uma flag 'is_email_valid')
    .filter(F.col("email").rlike(email_pattern))
    
    # Renomeia região tratada para o nome final
    .drop("regiao")
    .withColumnRenamed("regiao_clean", "regiao")
)

# 3. Deduplicação (Mesma lógica do seu notebook anterior)
window_spec = W.partitionBy("id_cliente").orderBy(F.col("ingestion_timestamp").desc())

df_dedup = (
    df_clientes
    .withColumn("row_number", F.row_number().over(window_spec))
    .filter(F.col("row_number") == 1)
    .drop("row_number")
)

# Seleção final e ordenação das colunas (snake_case implícito pois a origem já parece estar ok)
df_final = df_dedup.select(
    "id_cliente",
    "nome",
    "email",
    "regiao",
    "idade",
    "ingestion_timestamp"
)


In [0]:
df_final.write.mode("overwrite").saveAsTable("silver.dim_clientes")

In [0]:
print(f"Linhas finais: {df_final.count()}")
display(df_final.limit(10))

## Tabela: Custos

In [0]:
# Lendo a tabela e vizualizando os primeiros registros
df_custos_bronze = spark.table("bronze.custos")
print(f"{df_custos_bronze.count()} rows")
display(df_custos_bronze.limit(10))

In [0]:
# Removendo duplicatas
df_custos_bronze = df_custos_bronze.distinct()

# Limpando os dados, manténdo apenas números, vírgula e ponto
df_clean = df_custos_bronze.withColumn(
    "custo_limpo",
    regexp_replace(col("custo"), "[^0-9,\\.]", "") 
)

# Trocando vírgula por pontos 
df_clean = df_clean.withColumn(
    "custo_padronizado",
    regexp_replace(col("custo_limpo"), ",", ".")
)


# Alterando o tipo de dados de custo para decimal
df_clean = df_clean.withColumn(
    "custo_final",
    col("custo_padronizado").cast(DecimalType(18, 10))
)

# Padronizando nome das colunas com snake_case
df_clean = df_clean.withColumnRenamed("custo_final", "valor_custo")

df_custos_silver = df_clean.select(
    "id_custo",
    "id_chamado",
    "valor_custo"
)

# Atualizando tempo de ingenstão para camanda silver
df_custos_silver = df_custos_silver.withColumn("ingestion_timestamp", current_timestamp())


In [0]:
df_custos_silver.write.mode("overwrite").saveAsTable("silver.ft_custos")

In [0]:
print(f"{df_custos_silver.count()} rows")
display(df_custos_silver.limit(10))

## Tabela: Motivos

## Tabela: Pesquisa_Satisfação